# Getting Started with Prompt Engineering

This notebook contains examples and exercises to learning about prompt engineering.

We will be using the [Google AI Studio](https://aistudio.google.com/) for all examples.

---

## 1. Prompt Engineering Basics

Objectives
- Load the libraries
- Review the format
- Cover basic prompts
- Review common use cases

Below we are loading the necessary libraries, utilities, and configurations.

In [1]:
%%capture
# update or install the necessary libraries
!pip install --upgrade google-genai

In [3]:
import IPython
from google import genai
from google.genai import types
from google.colab import userdata

Load environment variables. You should create your own free Gemini API key in [Google AI Studio](https://aistudio.google.com/), Store the key in Colab Secrets(the 🔑 icon in the Colab left sidebar) as `GEMINI_API_KEY`. Do not paste any API key directly into the notebook.

In [4]:
# API configuration
gemini_api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=gemini_api_key)

In [5]:
MODEL_NAME = "gemini-3.1-flash-lite"


def set_open_params(
    model=MODEL_NAME,
    temperature=0.7,
    max_tokens=1024,
    top_p=1,
):
    """Bundle Gemini generation parameters into a dict."""
    return {
        "model": model,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "top_p": top_p,
    }


def get_completion(params, prompt):
    """Send a prompt to Gemini and return the generated text."""
    config = types.GenerateContentConfig(
        temperature=params["temperature"],
        top_p=params["top_p"],
        max_output_tokens=params["max_tokens"],
    )
    response = client.models.generate_content(
        model=params["model"],
        contents=prompt,
        config=config,
    )
    return (response.text or "").strip()


Basic prompt example:

In [40]:
# basic example
params = set_open_params()

prompt = "The sky is"

generated_text = get_completion(params, prompt)

IPython.display.Markdown(generated_text)

The sky is commonly described as **blue** during the day due to a phenomenon called Rayleigh scattering, where sunlight interacts with the Earth's atmosphere and scatters shorter blue wavelengths of light in every direction.

However, the appearance of the sky changes depending on several factors:

*   **At sunrise and sunset:** It often appears red, orange, or pink because the sunlight has to travel through more of the atmosphere, scattering away the blue light and leaving the longer wavelengths.
*   **At night:** It appears black because there is no direct sunlight to scatter.
*   **During cloudy weather:** It often looks gray or white because clouds scatter all wavelengths of light equally (known as Mie scattering).

Are you looking for a poetic description, a scientific explanation, or something else?

Try different temperature or top-p to compare results, and why do they differ or not?



In [41]:
params = set_open_params(temperature=0)
prompt = "The sky is"
response = get_completion(params, prompt)
IPython.display.Markdown(response)

The sky is often described as **blue** during the day due to a phenomenon called Rayleigh scattering, where the Earth's atmosphere scatters sunlight in all directions.

However, its appearance changes depending on several factors:

*   **Sunrise and Sunset:** The sky can turn shades of red, orange, pink, and purple because the sunlight has to travel through more of the atmosphere, scattering away the blue light and leaving the longer wavelengths.
*   **Night:** The sky appears black because there is no direct sunlight to scatter.
*   **Weather:** Clouds can make the sky appear white or gray, and storms can sometimes give it a dark, greenish, or bruised purple hue.
*   **Perspective:** From space, the sky is black because there is no atmosphere to scatter the light.

Are you looking for a scientific explanation, a poetic description, or something else?

In [45]:
params = set_open_params(top_p=0)
prompt = "The sky is"
response = get_completion(params, prompt)
IPython.display.Markdown(response)

The sky is often described as **blue** during the day due to a phenomenon called Rayleigh scattering, where the Earth's atmosphere scatters sunlight in all directions.

However, its appearance changes depending on several factors:

*   **Sunrise and Sunset:** The sky can turn shades of red, orange, pink, and purple because the sunlight has to travel through more of the atmosphere, scattering away the blue light and leaving the longer wavelengths.
*   **Night:** The sky appears black because there is no direct sunlight to scatter.
*   **Weather:** Clouds can make the sky appear white or gray, and storms can sometimes give it a dark, greenish, or bruised purple hue.
*   **Perspective:** From space, the sky is black because there is no atmosphere to scatter the light.

Are you looking for a scientific explanation, a poetic description, or something else?

<details>
<summary><b>Aside: how <code>temperature</code> and <code>top_p</code> actually shape the output</b> (click to expand)</summary>

This is a side note. It is useful for tuning the decoding side, but not the focus of this notebook. Skim or skip as you prefer. You can also use the *Text Summarization* prompt from §1.1 below to help you learn this part :)

Both parameters change the next-token distribution before sampling. They act on different parts of it.

**Temperature** rescales the whole distribution. At each step the model produces logits over the vocabulary. We divide the logits by `T` and apply softmax again. When `T < 1` the distribution gets sharper. The top tokens grab even more probability mass. When `T > 1` the distribution gets flatter. Low-probability tokens become viable. At `T = 0` the distribution collapses to a single point on the argmax. This is pure greedy decoding. At `T = 1` the original distribution is used as is.

**Top_p** (nucleus sampling) truncates the distribution instead. We sort tokens by probability. We keep the smallest set whose cumulative probability reaches `p`. The rest are discarded. The survivors are then renormalised. With `top_p = 1` nothing is cut. With `top_p = 0` only the argmax survives. This is also pure greedy decoding. Values like `0.9` or `0.95` chop off the long tail of unlikely tokens. The relative weights among the top tokens stay the same.

This explains what you saw earlier. `temperature = 0` and `top_p = 0` are two different ways to reach greedy decoding. So they give the same output. They behave differently at intermediate values. Temperature can revive unlikely tokens by flattening the curve. Top_p cannot. Once a token is truncated it is gone.

A reasonable default: leave one at its identity value (`temperature = 1` or `top_p = 1`) and tune the other. Setting both is fine, but the interaction is harder to reason about. On modern instruction-tuned models the gain over tuning just one is usually small.

For a longer treatment of decoding strategies, including top-k, min-p, and typical sampling, see Hugging Face's [*How to generate text*](https://huggingface.co/blog/how-to-generate) blog post.

</details>

### 1.1 Text Summarization

In [10]:
params = set_open_params(temperature=0.7)
prompt = """Antibiotics are a type of medication used to treat bacterial infections. They work by either killing the bacteria or preventing them from reproducing, allowing the body's immune system to fight off the infection. Antibiotics are usually taken orally in the form of pills, capsules, or liquid solutions, or sometimes administered intravenously. They are not effective against viral infections, and using them inappropriately can lead to antibiotic resistance.

Explain the above in one sentence:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Antibiotics are medications that treat bacterial infections by killing or inhibiting bacteria, but they are ineffective against viruses and must be used correctly to prevent antibiotic resistance.

Exercise: Instruct the model to explain the paragraph in one sentence like "I am 5". Do you see any differences?

**Example solution.** We change the instruction at the end of the prompt from a neutral *"in one sentence"* to *"in one sentence like I'm 5 years old"*. Adding the audience constraint pushes the model toward simpler vocabulary, shorter clauses, and concrete analogies, even though the temperature and context are unchanged. Try running both versions side-by-side: the ELI5 version typically replaces *"bacterial infections"* with *"germs that make you sick"* and *"antibiotic resistance"* with something like *"the germs learn to ignore the medicine"*.


In [11]:
params = set_open_params(temperature=0.7)
prompt = """Antibiotics are a type of medication used to treat bacterial infections. They work by either killing the bacteria or preventing them from reproducing, allowing the body's immune system to fight off the infection. Antibiotics are usually taken orally in the form of pills, capsules, or liquid solutions, or sometimes administered intravenously. They are not effective against viral infections, and using them inappropriately can lead to antibiotic resistance.

Explain the above in one sentence like I'm 5 years old:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)


Antibiotics are special medicines that fight off bad germs called bacteria, but they don't work on colds or the flu, so we must be careful only to use them when a doctor says we really need them.

### 1.2 Question Answering

In [12]:
params = set_open_params()
prompt = """Answer the question based on the context below. Keep the answer short and concise. Respond "Unsure about answer" if not sure about the answer.

Context: Teplizumab traces its roots to a New Jersey drug company called Ortho Pharmaceutical. There, scientists generated an early version of the antibody, dubbed OKT3. Originally sourced from mice, the molecule was able to bind to the surface of T cells and limit their cell-killing potential. In 1986, it was approved to help prevent organ rejection after kidney transplants, making it the first therapeutic antibody allowed for human use.

Question: What was OKT3 originally sourced from?

Answer:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)


Mice.

Context obtained from here: https://www.nature.com/articles/d41586-023-00400-x

Exercise: Edit prompt and get the model to respond that it isn't sure about the answer.

**Example solution.** The original prompt already tells the model to reply *"Unsure about answer"* when it cannot answer from the context. To actually trigger that fallback we just have to ask a question that genuinely is **not** answerable from the passage, e.g. the *current market price* of Teplizumab, which the context never mentions.


In [49]:
params = set_open_params(temperature=0)
prompt = """Answer the question based on the context below. Keep the answer short and concise. Respond "Unsure about answer" if not sure about the answer.

Context: Teplizumab traces its roots to a New Jersey drug company called Ortho Pharmaceutical. There, scientists generated an early version of the antibody, dubbed OKT3. Originally sourced from mice, the molecule was able to bind to the surface of T cells and limit their cell-killing potential. In 1986, it was approved to help prevent organ rejection after kidney transplants, making it the first therapeutic antibody allowed for human use.

Question: What is the current market price of Teplizumab?

Answer:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)


Unsure about answer

### 1.3 Text Classification

In [14]:
prompt = """Classify the text into neutral, negative or positive.

Text: I think the food was okay.

Sentiment:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Sentiment: Neutral

Exercise: Modify the prompt to instruct the model to provide an explanation to the answer selected.

**Example solution.** We add two things: (1) an explicit instruction asking for an explanation, and (2) a structured output format. Pinning down the format makes the response easier to parse downstream (e.g. for evaluation) and forces the model to commit to a label *before* rationalising, which reduces the risk of the label drifting to fit a verbose justification. The model should also point to specific evidence (here: the word *"okay"*) rather than handwave.


In [15]:
prompt = """Classify the text into one of: neutral, negative, or positive.
Then briefly explain your classification, citing specific words or phrases from the text.

Text: I think the food was okay.

Respond in exactly this format:
Sentiment: <label>
Explanation: <one or two sentences>"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)


Sentiment: neutral
Explanation: The word "okay" indicates a mediocre experience that is neither explicitly good nor bad, suggesting a middle-of-the-road opinion.

### 1.4 Role Playing

In [16]:
prompt = """The following is a conversation with an AI research assistant. The assistant tone is technical and scientific.

Human: Hello, who are you?
AI: Greeting! I am an AI research assistant. How can I help you today?
Human: Can you tell me about the creation of blackholes?
AI:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

The formation of a black hole is fundamentally a consequence of gravitational collapse, occurring when a massive object’s internal pressure is insufficient to counteract its own gravitational potential. This process is primarily understood through the lens of general relativity and stellar evolution.

The most common mechanism for black hole formation is the **core-collapse of a massive star**. The sequence of events is as follows:

1.  **Nuclear Exhaustion:** A star with a progenitor mass significantly greater than the Chandrasekhar limit (approximately 1.4 solar masses) undergoes successive stages of nuclear fusion. Once the core is converted into iron, fusion becomes endothermic, meaning it no longer generates the outward thermal pressure required to maintain hydrostatic equilibrium.
2.  **Gravitational Instability:** Without the outward radiation pressure, gravity dominates. The core undergoes a rapid, catastrophic collapse.
3.  **Supernova and Remnant:** If the remaining core mass exceeds the Tolman-Oppenheimer-Volkoff (TOV) limit—the theoretical maximum mass for a neutron star, typically cited between 2.1 and 3 solar masses—neutron degeneracy pressure is overcome. The core continues to collapse indefinitely.
4.  **Singularity Formation:** The matter is compressed into a region of infinite density and zero volume, known as a gravitational singularity. This is enclosed by an event horizon—a boundary in spacetime defined by the Schwarzschild radius ($R_s = 2GM/c^2$), beyond which the escape velocity exceeds the speed of light.

Beyond stellar-mass black holes, there are other theoretical and observed pathways:

*   **Primordial Black Holes:** These are hypothesized to have formed in the high-density environment of the early universe, shortly after the Big Bang, due to density fluctuations rather than stellar collapse.
*   **Direct Collapse:** In the early universe, massive gas clouds may have bypassed the star-formation phase entirely, collapsing directly into supermassive black holes.
*   **Supermassive Black Hole Growth:** While the origin of the supermassive black holes at the centers of galaxies remains a subject of active research, it is generally attributed to a combination of gas accretion, stellar mergers, and the hierarchical merging of smaller black holes over cosmological timescales.

Are you interested in the mathematical formalism of the Schwarzschild metric, or perhaps the thermodynamic properties of these objects, such as Hawking radiation?

Exercise: Modify the prompt to instruct the model to keep AI responses concise and short.

**Example solution.** Two changes work together here: (1) we add a length constraint to the system description (*"keep every response under two sentences"*), and (2) we shorten the assistant's in-prompt greeting so the few-shot pattern itself stays concise. Models tend to imitate the verbosity they see in the conversation history, so making the example reply terse is as important as stating the rule explicitly. If the assistant still over-explains, tighten further to *"answer in one sentence"*.


In [17]:
prompt = """The following is a conversation with an AI research assistant. The assistant tone is technical and scientific. The assistant keeps every response concise — no more than two sentences — and avoids unnecessary preamble.

Human: Hello, who are you?
AI: AI research assistant. How can I help?
Human: Can you tell me about the creation of blackholes?
AI:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)


Black holes form when massive stars exhaust their nuclear fuel, leading to gravitational collapse under their own mass. This process compresses the stellar core into a singularity, creating a region where gravity is so intense that not even light can escape.

### 1.5 Code Generation

In [18]:
prompt = "\"\"\"\nTable departments, columns = [DepartmentId, DepartmentName]\nTable students, columns = [DepartmentId, StudentId, StudentName]\nCreate a MySQL query for all students in the Computer Science Department\n\"\"\""

response = get_completion(params, prompt)
IPython.display.Markdown(response)


To retrieve all students in the "Computer Science" department, you need to perform a `JOIN` between the two tables on the `DepartmentId` column.

```sql
SELECT 
    s.StudentId, 
    s.StudentName
FROM students s
JOIN departments d ON s.DepartmentId = d.DepartmentId
WHERE d.DepartmentName = 'Computer Science';
```

### Explanation:
*   **`JOIN`**: Connects the `students` table with the `departments` table using the common `DepartmentId` field.
*   **`ON s.DepartmentId = d.DepartmentId`**: Specifies the relationship between the two tables.
*   **`WHERE d.DepartmentName = 'Computer Science'`**: Filters the results to only include students belonging to the specified department.

### 1.6 Reasoning

In [19]:
prompt = """The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1.

Solve by breaking the problem into steps. First, identify the odd numbers, add them, and indicate whether the result is odd or even."""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

To determine if the sum of the odd numbers in the group is even, we will follow your requested steps:

**Step 1: Identify the odd numbers**
The odd numbers in the group (15, 32, 5, 13, 82, 7, 1) are:
**15, 5, 13, 7, and 1**

**Step 2: Add the odd numbers**
15 + 5 + 13 + 7 + 1 = **41**

**Step 3: Indicate whether the result is odd or even**
The result, 41, is an **odd number**.

**Conclusion:**
The statement that the odd numbers add up to an even number is **false**.

Exercise: Improve the prompt to have a better structure and output format.

**Example solution.** The original prompt asks the model to *"break the problem into steps"* but doesn't specify what those steps look like, so the output format drifts. We fix this with three changes: (1) name the steps explicitly (`Step 1 / Step 2 / Step 3`), (2) say what each step must produce, and (3) require a final line with the format `Final Answer: <True if sum is even, False if sum is odd>`. This kind of contract makes the chain-of-thought *evaluable*. A downstream script can regex-match the final line and check correctness without parsing prose.


In [20]:
prompt = """Determine whether the odd numbers in the group below add up to an even number.

Group: 15, 32, 5, 13, 82, 7, 1

Solve by following this exact format:

Step 1 — Identify odd numbers: <comma-separated list>
Step 2 — Sum them: <show the addition, e.g. "15 + 5 + 13 + 7 + 1 = ...">
Step 3 — Parity of sum: <"even" or "odd">

Final Answer: <True if sum is even, False if sum is odd>"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)


Step 1 — Identify odd numbers: 15, 5, 13, 7, 1
Step 2 — Sum them: 15 + 5 + 13 + 7 + 1 = 41
Step 3 — Parity of sum: odd

Final Answer: False

---

## 2. Advanced Prompting Techniques

Objectives:

- Cover more advanced techniques for prompting: few-shot, chain-of-thoughts,...

### 2.2 Few-shot prompts

In [21]:
prompt = """The odd numbers in this group add up to an even number: 4, 8, 9, 15, 12, 2, 1.
A: The answer is False.

The odd numbers in this group add up to an even number: 17,  10, 19, 4, 8, 12, 24.
A: The answer is True.

The odd numbers in this group add up to an even number: 16,  11, 14, 4, 8, 13, 24.
A: The answer is True.

The odd numbers in this group add up to an even number: 17,  9, 10, 12, 13, 4, 2.
A: The answer is False.

The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1.
A:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

The answer is False.

**Reasoning:**
The odd numbers in the group are 15, 5, 13, 7, and 1.
15 + 5 + 13 + 7 + 1 = 41.
Since 41 is an odd number, the statement is False.

### 2.3 Chain-of-Thought (CoT) Prompting

In [22]:
prompt = """The odd numbers in this group add up to an even number: 4, 8, 9, 15, 12, 2, 1.
A: Adding all the odd numbers (9, 15, 1) gives 25. The answer is False.

The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1.
A:"""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Adding all the odd numbers (15, 5, 13, 7, 1) gives 41. The answer is False.

### 2.4 Zero-shot CoT

In [23]:
prompt = """I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. How many apples did I remain with?

Let's think step by step."""

response = get_completion(params, prompt)
IPython.display.Markdown(response)

Here is the step-by-step breakdown:

1.  **Starting amount:** You bought 10 apples.
2.  **Giving away apples:** You gave 2 to the neighbor and 2 to the repairman (10 - 2 - 2 = 6). You had 6 apples left.
3.  **Buying more:** You bought 5 more apples (6 + 5 = 11). You had 11 apples.
4.  **Eating an apple:** You ate 1 apple (11 - 1 = 10).

You remained with **10** apples.

---

## Challenge 1: Advanced Few-Shot Prompting

**Technique:** Provide multiple high-quality exemplars so the model learns your preferred style of solution.

**Input Question:**  
Find the domain of  
*f(x) = √(x² − 5x + 6)*

**Task:**  
Design a prompt with **3** solved examples of “find the domain of a square-root function”

1. States the function.  
2. Explains the inequality step by step.  
3. Gives the final domain in interval notation.

> **Example prompt**  
> ```text
> Example 1:
> Q: Find the domain of f(x) = √(2x − 4).
> A:
>  1. Require 2x − 4 ≥ 0  
>  2. 2x ≥ 4 ⇒ x ≥ 2  
>  3. Domain: [2, ∞)
>
> Example 2:
> Q: Find the domain of f(x) = √(5 − x).
> A:
>  1. Require 5 − x ≥ 0  
>  2. −x ≥ −5 ⇒ x ≤ 5  
>  3. Domain: (−∞, 5]
>
> Example 3:
> Q: Find the domain of f(x) = √(x + 1).
> A:
>  1. Require x + 1 ≥ 0  
>  2. x ≥ −1  
>  3. Domain: [−1, ∞)
>
> **Now you:**
> Q: Find the domain of f(x) = √(3x − 2).
> A:
> ```

In [24]:
# Example solution
prompt = """ Example 1:
Q: Find the domain of f(x) = √(2x − 4).
A:
 1. Require 2x − 4 ≥ 0
 2. 2x ≥ 4 ⇒ x ≥ 2
 3. Domain: [2, ∞)

Example 2:
Q: Find the domain of f(x) = √(5 − x).
A:
 1. Require 5 − x ≥ 0
 2. −x ≥ −5 ⇒ x ≤ 5
 3. Domain: (−∞, 5]

Example 3:
Q: Find the domain of f(x) = √(x + 1).
A:
 1. Require x + 1 ≥ 0
 2. x ≥ −1
 3. Domain: [−1, ∞)

**Now you:**
Q: Find the domain of f(x) = √(x² − 5x + 6).
A:
"""
response = get_completion(params, prompt)
IPython.display.Markdown(response)


1. Require $x^2 - 5x + 6 \geq 0$
2. Factor the quadratic: $(x - 2)(x - 3) \geq 0$
3. Determine the intervals: The roots are $x = 2$ and $x = 3$. Testing the intervals $(-\infty, 2]$, $[2, 3]$, and $[3, \infty)$, the expression is non-negative when $x \leq 2$ or $x \geq 3$.
4. Domain: $(-\infty, 2] \cup [3, \infty)$

---

## Challenge 2: Tree-of-Thought Prompting

**Technique:** Encourage the model to **branch** its reasoning, exploring multiple sub-steps in parallel before committing to an answer.

**Input Question:**  
Find the domain of  
*f(x) = √(9 − x²) / (x − 2)*

**Task:**  
Write a prompt that tells the model to:

1. **List** all conditions (branches) that must be satisfied.  
2. **Evaluate** each branch separately.  
3. **Combine** them for the final domain.

> **Example prompt**  
> ```text
> Solve for the domain of f(x) = √(3 − x²) using a Tree-of-Thought:
>
> **Step 1**: List branches  
> - Branch A: 3 − x² ≥ 0  
>
> **Step 2**: Explore each branch  
> > *Branch A1*: Solve 3 − x² ≥ 0  
> > - x² ≤ 3 ⇒ −√3 ≤ x ≤ √3  
>
> **Step 3**: Combine results  
> - Only one branch, so domain is [−√3, √3]
>
> **Answer**: [−√3, √3]
> ```


In [25]:
# Example solution
prompt = """ **Question**: Solve for the domain of f(x) = √(9 − x²) / (x − 2) using a Tree-of-Thought:

**Instructions**:
1. List all conditions (branches) that must be satisfied.
2. Evaluate each branch separately.
3. Combine them for the final domain.

**Answer**:
"""
response = get_completion(params, prompt)
IPython.display.Markdown(response)

To solve for the domain of the function $f(x) = \frac{\sqrt{9 - x^2}}{x - 2}$, we will break the problem down into logical branches.

### 1. List all conditions (branches)
For the function to be defined, two primary conditions must be met:
*   **Branch A (Radicand condition):** The expression inside the square root must be non-negative because the square root of a negative number is not a real number.
    *   $9 - x^2 \geq 0$
*   **Branch B (Denominator condition):** The denominator of a fraction cannot be zero, as division by zero is undefined.
    *   $x - 2 \neq 0$

---

### 2. Evaluate each branch separately

**Evaluating Branch A ($9 - x^2 \geq 0$):**
*   $9 \geq x^2$
*   Taking the square root of both sides, we get $|x| \leq 3$.
*   This inequality represents the interval: **$[-3, 3]$**.

**Evaluating Branch B ($x - 2 \neq 0$):**
*   $x \neq 2$
*   This means that the value $x = 2$ must be excluded from any set of real numbers.

---

### 3. Combine for the final domain

To find the final domain, we take the intersection of the results from Branch A and Branch B:
*   Start with the interval from Branch A: $[-3, 3]$.
*   Apply the restriction from Branch B: Remove $x = 2$ from that interval.

This splits the interval $[-3, 3]$ into two parts:
1.  From $-3$ up to (but not including) $2$: $[-3, 2)$
2.  From $2$ (not including) up to $3$: $(2, 3]$

**Final Domain:**
In interval notation, the domain is:
**$[-3, 2) \cup (2, 3]$**

---

## Challenge 3: Program-Aided Language Models

**Technique:** Instruct the model to **call external code** (e.g., Python/Sympy) to handle the algebra, then summarize the result.

**Input Question:**  
Find the domain of  
*f(x) = √(x + 5) / √(10 − x)*

**Task:**  
Prompt the model to:

1. Write a short Python script using Sympy to solve an inequality.  
2. Run it mentally (or via a stub) and then present the domain.

> **Example prompt**  
> ```text
> **Question**: Find the domain of f(x) = √(3x − 2).
>
> **Instructions**:
> 1. Write Python code with Sympy to solve 3x − 2 ≥ 0.  
> 2. Execute the code and capture the output.  
> 3. State the domain in interval notation.
>
> ```python
> import sympy as sp
> x = sp.symbols('x')
> expr = 3*x - 2
> solution = sp.solve_univariate_inequality(expr >= 0, x)
> print(solution)
> ```
>

In [51]:
# Example solution
prompt = """ **Question**: Find the domain of f(x) = √(x + 5) / √(10 − x).

**Instructions**:
1. Identify all constraints the domain must satisfy.
2. For each constraint, write Sympy code that solves it. Show the code.
3. State the output you would expect from running each snippet.
4. Combine the constraints and give the final domain in interval notation.

**Answer**:
"""
response = get_completion(params, prompt)
IPython.display.Markdown(response)

To find the domain of the function $f(x) = \frac{\sqrt{x + 5}}{\sqrt{10 - x}}$, we must ensure that the expression is defined within the real number system.

### 1. Identify Constraints
There are two primary constraints:
*   **Constraint 1 (Numerator):** The expression inside the square root in the numerator must be non-negative: $x + 5 \geq 0$.
*   **Constraint 2 (Denominator):** The expression inside the square root in the denominator must be strictly positive (it cannot be zero because division by zero is undefined): $10 - x > 0$.

---

### 2. SymPy Code
We use SymPy to solve these inequalities.

```python
from sympy import symbols, solve_inequality, GreaterThan, LessThan, StrictGreaterThan

x = symbols('x')

# Constraint 1: x + 5 >= 0
constraint1 = x + 5 >= 0
sol1 = solve_inequality(constraint1, x)

# Constraint 2: 10 - x > 0
constraint2 = 10 - x > 0
sol2 = solve_inequality(constraint2, x)

print(f"Constraint 1 solution: {sol1}")
print(f"Constraint 2 solution: {sol2}")
```

---

### 3. Expected Output
Running the code above will yield:
*   **Constraint 1 solution:** `x >= -5` (or in interval notation: $[-5, \infty)$)
*   **Constraint 2 solution:** `x < 10` (or in interval notation: $(-\infty, 10)$)

---

### 4. Final Domain
To find the final domain, we must find the intersection of the two sets:
*   $x \geq -5$
*   $x < 10$

The intersection of these two conditions is $-5 \leq x < 10$.

**Final Domain in Interval Notation:**
**$[-5, 10)$**